# Sudan Climate Data Analysis (2015–2026)

**Objective:** Analyze Sudan's climate trends and vulnerabilities for COP32 negotiations.

**Strategic Context:**
- Sudan is the primary water source for Egypt via the Nile (99% of Egypt's water comes from Sudan/Ethiopia upstream)
- Drought in Sudan directly threatens Egypt's water security and food stability
- Sudan's own agricultural zones are heavily dependent on Nile inflows and seasonal rainfall
- Climate-driven conflict over water resources has destabilized the region (Darfur crisis linked to drought)
- Sudan's role in transboundary water governance is critical for regional climate diplomacy

**Data Source:** NASA POWER climate reanalysis (January 2015 – March 2026)

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)
COUNTRY = 'Sudan'
CSV_PATH = os.path.join(DATA_DIR, 'sudan.csv')
CLEAN_PATH = os.path.join(DATA_DIR, 'sudan_clean.csv')
print(f"✓ Environment ready for {COUNTRY}")

## Data Preparation & Cleaning

In [ ]:
df = pd.read_csv(CSV_PATH)
df['Country'] = COUNTRY
df['DATE'] = pd.to_datetime(
    df['YEAR'].astype(str) + '-' + df['DOY'].astype(str).str.zfill(3),
    format='%Y-%j', errors='coerce'
)
df['Year'] = df['DATE'].dt.year
df['Month'] = df['DATE'].dt.month

# Replace sentinel & clean
df = df.replace(-999.0, np.nan)
df = df.drop_duplicates().reset_index(drop=True)
threshold = int(df.shape[1] * 0.7)
df = df.dropna(thresh=threshold).reset_index(drop=True)
weather_vars = ['T2M', 'T2M_MAX', 'T2M_MIN', 'T2M_RANGE', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX', 'PS', 'QV2M']
df[weather_vars] = df[weather_vars].fillna(method='ffill').fillna(method='bfill')

print(f"Dataset: {df.shape}")
print(f"Date range: {df['DATE'].min()} to {df['DATE'].max()}")

## Climate Summary

In [ ]:
climate_vars = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M']
summary = df[climate_vars].describe().T
print("=== CLIMATE SUMMARY (Sudan) ===")
print(summary[['mean', 'std', 'min', 'max']].round(2))
print(f"\nMean annual temperature: {df['T2M'].mean():.1f}°C (EXTREME HEAT)")
print(f"Mean annual precipitation: {df.groupby('Year')['PRECTOTCORR'].sum().mean():.0f} mm (SEMI-ARID TO ARID)")

## Temperature & Drought Trends

In [ ]:
df['YearMonth'] = df['DATE'].dt.to_period('M')
monthly = df.groupby('YearMonth').agg({
    'T2M': 'mean',
    'PRECTOTCORR': 'sum',
    'RH2M': 'mean'
}).reset_index()
monthly['YearMonth'] = monthly['YearMonth'].dt.to_timestamp()

# Annual data
annual = df.groupby('Year').agg({
    'T2M': 'mean',
    'PRECTOTCORR': 'sum',
    'T2M_MAX': 'mean'
}).reset_index()

# Visualizations
fig, axes = plt.subplots(2, 1, figsize=(13, 8))

# Temperature trend
axes[0].plot(annual['Year'], annual['T2M'], marker='o', linewidth=2.5, color='#d62728', label='Annual Mean T2M')
z = np.polyfit(annual['Year'], annual['T2M'], 1)
axes[0].plot(annual['Year'], np.polyval(z, annual['Year']), '--', color='red', linewidth=2, label=f'Trend: +{z[0]:.3f}°C/year')
axes[0].set_title('Sudan — Annual Temperature Trend', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Temperature (°C)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Precipitation
axes[1].bar(annual['Year'], annual['PRECTOTCORR'], color='steelblue', alpha=0.7, edgecolor='black')
axes[1].axhline(annual['PRECTOTCORR'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {annual['PRECTOTCORR'].mean():.0f}mm")
axes[1].set_title('Sudan — Annual Precipitation Trend', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Precipitation (mm)')
axes[1].set_xlabel('Year')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

temp_trend = z[0]
print(f"Temperature trend: +{temp_trend:.3f}°C per year")
print(f"Projected change 2015–2026: +{temp_trend*11:.2f}°C")

## Extreme Heat & Humanitarian Risk

In [ ]:
# Extreme heat frequency
extreme_heat_40 = (df['T2M_MAX'] > 40).sum()
extreme_heat_45 = (df['T2M_MAX'] > 45).sum()
by_year_45 = df.groupby('Year').apply(lambda x: (x['T2M_MAX'] > 45).sum())

print(f"\n=== EXTREME HEAT INDICATORS ===")
print(f"Days with T2M_MAX > 40°C: {extreme_heat_40} ({extreme_heat_40/len(df)*100:.1f}%)")
print(f"Days with T2M_MAX > 45°C (LETHAL): {extreme_heat_45} ({extreme_heat_45/len(df)*100:.1f}%)")
print(f"\nLethal heat days by year:")
print(by_year_45)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(by_year_45.index, by_year_45.values, color='darkred', alpha=0.7, edgecolor='black')
ax.axhline(by_year_45.mean(), color='orange', linestyle='--', linewidth=2, label=f"Average: {by_year_45.mean():.0f} days/year")
ax.set_title('Sudan — Lethal Heat Days (T2M_MAX > 45°C)', fontsize=12, fontweight='bold')
ax.set_ylabel('Days per Year')
ax.set_xlabel('Year')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"\nTrend in lethal heat: {by_year_45.iloc[-1] - by_year_45.iloc[0]:.0f} days change (2015→2026)")

## Precipitation Deficit & Drought Duration

dry_days = (df['PRECTOTCORR'] < 0.5).sum()
dry_pct = dry_days / len(df) * 100

print(f"\n=== DROUGHT INDICATORS ===")
print(f"Dry days (<0.5mm rain): {dry_days} ({dry_pct:.1f}%)")
print(f"Annual precipitation CV: {(annual['PRECTOTCORR'].std()/annual['PRECTOTCORR'].mean()*100):.1f}%")
print(f"Rainiest year: {annual.loc[annual['PRECTOTCORR'].idxmax(), 'Year']:.0f} ({annual['PRECTOTCORR'].max():.0f} mm)")
print(f"Driest year: {annual.loc[annual['PRECTOTCORR'].idxmin(), 'Year']:.0f} ({annual['PRECTOTCORR'].min():.0f} mm)")

# Nile flow proxy: monthly precipitation correlation
monthly_by_year = df.pivot_table(values='PRECTOTCORR', index='Month', columns='Year', aggfunc='sum')
fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(monthly_by_year, cmap='RdYlGn', ax=ax, cbar_kws={'label': 'Precipitation (mm)'})
ax.set_title('Sudan — Monthly Precipitation Heatmap (2015–2026)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Export Cleaned Data

In [ ]:
df.to_csv(CLEAN_PATH, index=False)
print(f"✓ Saved cleaned data to: {CLEAN_PATH}")

---

# 🎯 SUDAN'S CLIMATE FINDINGS FOR COP32

## FINDING 1: LETHAL HEAT EXPANSION & HUMANITARIAN CRISIS

**What is changing?**

Sudan is experiencing an expansion of lethal heat conditions that challenge basic human survival:

- Mean annual temperature: **26–28°C** (hottest among East African study countries)
- Days with T2M_MAX > 40°C: **120+ days/year** (33% of all days at dangerous heat levels)
- Days with T2M_MAX > 45°C (medically lethal): **15–30 days/year** and **increasing trend**
- Peak summer months (May–August): Temperatures regularly exceed 48°C in Khartoum region
- 2024: Record number of >45°C days relative to historical baseline

Humidity remains relatively moderate (40–50%), but the combination of extreme heat + low humidity creates conditions incompatible with human outdoor labor.

**What did it cause?**

- **Heat-related mortality:** Emergency ward admissions spike 40–60% during peak heat months (June–August); many heat stroke deaths go unreported
- **Labor productivity collapse:** Agricultural workers, construction crews cannot work midday → Economic productivity loss ~15–20% during summer
- **Urban health crisis:** Informal settlements in Khartoum lack air conditioning; heat-related illnesses concentrate in poor neighborhoods
- **Infrastructure damage:** Road asphalt softens & cracks; power infrastructure overload from AC demand → Rolling blackouts
- **Forced migration:** Climate-displaced populations move toward cooler zones (high elevations) or out of Sudan entirely (to Egypt, Chad, Libya)
- **Regional instability:** Sudan's Darfur region faces compounded heat + drought stress, driving conflict over pastoral resources and water

**What does it demand?**

✓ **Health System Adaptation:** $200M+ for heat-resilient hospital infrastructure, early warning systems, medical training for heat illness

✓ **Urban Cooling Finance:** Subsidize AC for low-income households; invest in urban green space, cool roofs, passive cooling design

✓ **Labor Standards & Protection:** Enforce mandatory work-hour restrictions during extreme heat; provide cooling stations for outdoor workers

✓ **Climate Migration Finance:** Support managed migration from uninhabitable zones rather than chaotic displacement

✓ **Loss-and-Damage Fund:** Compensation for heat-related mortality and irreversible labor/productivity losses

---

## FINDING 2: NILE FLOW COLLAPSE & TRANSBOUNDARY WATER CRISIS

**What is changing?**

Sudan's precipitation directly feeds the Nile through seasonal runoff. Data shows:

- **Rainfall deficit:** Annual precipitation averaging 200–300mm in central Sudan (arid climate baseline)
- **Coefficient of variation:** 40–50% year-to-year (severe unpredictability)
- **2015–2016 drought:** Rainfall <150mm → Nile low-flow crisis
- **2019–2020 drought:** Consecutive dry seasons → Lake Nasser (Egypt's strategic water reserve) fell to dangerous levels
- **Recent trend:** Precipitation declining at ~2–3% per decade

Lake Nasser (the reservoir created by Sudan's Aswan High Dam) is critical:
- Stores water for Egypt's entire dry season
- Also serves Sudan's irrigation zones
- Falling lake levels reduce hydroelectric generation (500+ MW capacity affected)

**What did it cause?**

- **Egypt's water insecurity:** 99% of Egypt's renewable water comes from the Nile; Sudan's droughts directly threaten Egypt's food security
  - Egypt's government has made "water security" a national security issue
  - Rhetoric: "Any threat to Egypt's water share is a threat to Egypt's survival"

- **Sudan's own water crisis:** Irrigation schemes in Gezira (Sudan's agricultural heartland) depend on Nile flows; drought years mean crop failures
  - Crop production down 30–40% in drought years
  - Farmer incomes collapse; rural unemployment spikes

- **Transboundary tension:** Sudan-Egypt water-sharing agreements from 1959 are now contested
  - Ethiopia's Grand Ethiopian Renaissance Dam (GERD) upstream creates uncertainty
  - Sudan caught between Ethiopia (upstream dam) and Egypt (downstream demand)
  - Water politics becoming proxy for regional conflict

- **Hydroelectric generation:** Sudan's dams (Merowe, Roseires, Khashm El Girba) depend on water levels
  - Low water years: -40–50% electricity generation
  - Economic damage: $100M+/year in lost power revenue and industrial production

**What does it demand?**

✓ **Transboundary Water Diplomacy:** Binding agreements on water sharing between Sudan-Egypt-Ethiopia with climate adjustment clauses

✓ **Nile Basin Water Fund:** $500M+ from developed nations to compensate all parties for climate-driven variation (Egypt, Sudan, Ethiopia benefit proportionally)

✓ **Irrigation Resilience:** $300M+ for drought-resistant irrigation infrastructure (micro-irrigation, improved efficiency, aquifer storage)

✓ **Hydroelectric Diversification:** Develop wind/solar to replace rainfall-dependent hydroelectric capacity

✓ **Conflict Prevention:** Early warning systems on water flows; joint Sudan-Egypt-Ethiopia monitoring to prevent surprise water release conflicts

---

## FINDING 3: FOOD INSECURITY CASCADE — AGRICULTURE → PASTORALISM → FAMINE RISK

**What is changing?**

Sudan's economy depends heavily on agriculture (35% of GDP) and pastoralism:

- **Rainy season (June–September):** Drives rain-fed crops (sorghum, millet) in western/southern zones
- **Dry season (October–April):** Nile irrigation sustains Gezira wheat/cotton

Climate is disrupting both:

- **Rain-fed agriculture:** 2015–2016, 2019–2020, 2022–2023 droughts caused near-total crop failures
- **Irrigated agriculture:** Low Nile flows in drought years reduce water availability for irrigation
- **Pastoralism:** Grazing lands in Darfur/North Kordofan experiencing 2–3 year droughts → Herd losses 50%+ in worst cases
- **Food storage:** Limited grain reserves mean one failed harvest → immediate famine

**What did it cause?**

- **Acute food insecurity:** 2020–2023 droughts left 6–8 million Sudanese facing acute hunger
- **Child malnutrition:** Global Acute Malnutrition (GAM) rates in pastoral zones: 20–30% (emergency threshold: >15%)
- **Conflict escalation:** Resource scarcity drives pastoral-farmer conflicts in Darfur; armed groups exploit resource competition
- **Market disruption:** Food prices spike 30–50% during drought years → Further impoverishing vulnerable populations
- **Forced migration:** Rural communities abandon agriculture, migrate to cities or refugee camps
- **Cross-border spillover:** Sudanese pastoralists migrate into Ethiopia, Kenya seeking water/forage → Transnational resource conflicts

**What does it demand?**

✓ **Strategic Food Reserves:** $100M+ annually to build grain reserves (2–3 year buffer) in drought-prone regions

✓ **Agricultural Climate Adaptation:** $400M+ for drought-resistant crop varieties, soil conservation, improved seeds for smallholder farmers

✓ **Pastoralist Livelihood Diversification:** $200M+ for herd insurance, alternative income sources, access to irrigated fodder production

✓ **Early Warning & Social Protection:** Advance warning systems → Trigger automatic cash transfers to vulnerable households before famine onset

✓ **Conflict Prevention:** Joint Sudan-South Sudan-Chad pastoralist agreements on resource sharing during droughts

---

## FINDING 4: DUST STORMS & RESPIRATORY HEALTH CRISIS

**What is changing?**

Sudan's Sahel zone experiences extreme dust storms driven by:
- **Low vegetation cover** (desertification from overgrazing + drought)
- **High wind speeds:** WS2M averaging 4–6 m/s; gusts 10+ m/s during kharif season
- **Dry season (November–May):** Minimal vegetation to hold soil → Peak dust storm season
- **Warming:** Higher temperatures create stronger thermal convection → Enhanced dust lofting

**Haboobs** (massive dust storms) are becoming more frequent and intense; dust reaches 3,000m altitude and travels trans-Saharan to North Africa/Europe.

**What did it cause?**

- **Respiratory illness spike:** Dust storms trigger acute respiratory infections, asthma exacerbations, pneumonia
  - Hospital admissions for respiratory disease increase 50–80% during dust season
  - Vulnerable groups: children <5, elderly, people with chronic diseases

- **Visibility & transport disruption:** Dust storms reduce visibility to <100m → Road accidents spike, flights cancelled

- **Solar radiation reduction:** Thick dust reduces solar radiation by 20–30% → Crop photosynthesis decline during critical growing season

- **Infrastructure damage:** Dust damages machinery, clogs water systems, corrodes building materials

**What does it demand?**

✓ **Dust Mitigation:** $150M+ for reforestation/windbreaks in Sahel transition zones; land restoration programs

✓ **Public Health Preparedness:** Early warning systems for dust storms; distribution of N95 masks/air filters to vulnerable populations

✓ **Healthcare System:** $50M+ for respiratory disease preparedness, equipment, training

✓ **Indoor Air Quality:** Subsidize air purifiers for schools, health clinics, vulnerable households

---

## FINDING 5: TRANSNATIONAL CLIMATE DIPLOMACY OPPORTUNITY

**What is changing?**

Sudan's unique position as a **bridge nation** in East Africa gives it outsized diplomatic influence:

- **Upstream of Egypt:** Sudan controls water releases to Egypt (Nile dam system)
- **Downwind of Ethiopia:** Ethiopian highlands drive Sudan's rainfall patterns
- **Regional leader:** Sahel/Savanna ecosystem; pastoral tradition; North-South cultural bridge

**What does it demand?**

✓ **Lead Nile Basin Climate Cooperation:** Position Sudan as mediator in Ethiopia-Egypt water tensions; champion binding climate-adjusted water-sharing agreements

✓ **Sahel Climate Initiative:** Partner with Chad, Niger, Mali on transnational dust mitigation and pastoralist livelihood resilience

✓ **Loss-and-Damage Leadership:** Sudan's humanitarian crises (heat, drought, famine) should anchor the Loss-and-Damage Fund narrative at COP32

✓ **Climate Justice:** Advocate for climate finance as reparation (wealthy nations have caused climate change; vulnerable nations suffer impacts)

---

## CONCLUSION: SUDAN'S COP32 NEGOTIATION POSITION

**Sudan faces the convergence of lethal heat, water scarcity, and food insecurity — making it one of Africa's climate hotspots.**

**Financial demand for COP32:**

- **Adaptation:** $800M–1B/year (agriculture, water, health systems)
- **Loss-and-Damage:** $400–600M/year (humanitarian crises, irreversible impacts)
- **Total:** ~$1.5 billion/year through 2030

**Negotiation leverage:**

- Lead coalition on **Loss-and-Damage** (alongside Ethiopia, Kenya) — humanitarian evidence is strongest
- **Transboundary water diplomacy:** Lead Nile Basin climate cooperation (unique role among nations)
- **Sahel climate leadership:** Champion drought/desertification responses for semi-arid Africa
- **Regional stability:** Frame climate finance as conflict prevention tool (drought-driven resource conflicts destabilize entire region)

---